# Experiment 05A–05B: independent-validation protocol audit

**Frozen before any independent inference or performance inspection.**

This notebook audits the protocol, documented dataset exposure, fixed acquisition ladder, primary hypotheses, development/calibration partitions and execution barriers. It contains no TESTIMAGES pixels and computes no reconstruction or selection performance.

The machine-readable protocol is authoritative. “No documented exposure found” is not a claim that checkpoint-level non-exposure has been proved.

In [1]:
from pathlib import Path
import csv
import hashlib
import importlib.util
import json
import pandas as pd
from IPython.display import display

ROOT = next(
    candidate for candidate in (Path.cwd(), Path.cwd().parent)
    if (candidate / 'experiments/independent_05/protocol_freeze.json').exists()
)
PROTOCOL_PATH = ROOT / 'experiments/independent_05/protocol_freeze.json'
AUDIT_PATH = ROOT / 'experiments/independent_05/dataset_exposure_audit.csv'
RECEIPT_PATH = ROOT / 'experiments/independent_05/freeze_receipt.json'
VALIDATOR_PATH = ROOT / 'scripts/validate_independent_05_protocol.py'

protocol = json.loads(PROTOCOL_PATH.read_text())
audit = pd.read_csv(AUDIT_PATH, keep_default_na=False)
receipt = json.loads(RECEIPT_PATH.read_text())

print(f"Experiment: {protocol['experiment_id']}")
print(f"Frozen: {protocol['frozen_at_utc']}")
print(f"Role: {protocol['role']}")
print(f"Independent results inspected: {protocol['test_results_inspected']}")
print(f"Independent run authorized: {protocol['independent_test_run_authorized']}")

Experiment: independent_05
Frozen: 2026-09-21T22:47:41Z
Role: intermediate_independent_confirmation
Independent results inspected: False
Independent run authorized: False


## 1. Claim boundary and present status

The protocol is an intermediate independent confirmation with a 40-source strict cohort. It does not meet the separate ≥100-source definitive gate and cannot establish novelty or prove checkpoint-training exclusion.

In [2]:
boundary = pd.DataFrame([
    {'item': key, 'value': value}
    for key, value in protocol['claim_boundary'].items()
])
status = pd.DataFrame([
    {'readiness_item': key, 'value': value}
    for key, value in protocol['current_status'].items()
])
display(boundary)
display(status)

,item,value
0,independent_source_evaluation_if_completed,True
1,checkpoint_training_overlap_proved_absent,False
2,reason_overlap_not_proved,The pinned publications and repositories name ...
3,calibrated_reliability_if_completed,"Limited to the frozen corruptions, model, scor..."
4,novelty_established,False
5,definitive_100_source_gate_satisfied,False
6,reason_100_source_gate_not_satisfied,The strict primary cohort has 40 natural-image...


,readiness_item,value
0,protocol_frozen,True
1,documented_exposure_audit_complete,True
2,checkpoint_training_overlap_proved_absent,False
3,independent_dataset_downloaded_and_hashed,False
4,near_duplicate_audit_complete,False
5,trained_comparator_ready,False
6,calibration_mappings_ready,False
7,engineering_canary_passed,False
8,independent_test_run,False
9,next_stage,05C_data_receipt_implementation_and_developmen...


## 2. Documented checkpoint exposure audit

Rows are conservative. Documented training overlap causes exclusion. Documented author benchmark exposure also excludes a dataset from the strict primary cohort. Absence from the reviewed pinned sources is recorded only as absence of documentation.

In [3]:
exposure_view = audit[[
    'dataset_or_family',
    'documented_fbc_nn_training',
    'documented_fbc_nn_benchmark',
    'documented_drunet_training',
    'documented_drunet_benchmark',
    'primary_test_eligibility',
]].copy()
display(exposure_view)

eligible = audit.loc[audit.primary_test_eligibility.eq('eligible with caveat')]
assert eligible.dataset_or_family.tolist() == ['TESTIMAGES/SAMPLING']
assert audit.loc[audit.dataset_or_family.eq('DIV2K'), 'primary_test_eligibility'].item() == 'exclude'
assert audit.loc[audit.dataset_or_family.eq('Flickr2K'), 'primary_test_eligibility'].item() == 'exclude'
print('PASS — exactly one strict primary cohort is eligible, with the stated caveat.')

,dataset_or_family,documented_fbc_nn_training,documented_fbc_nn_benchmark,documented_drunet_training,documented_drunet_benchmark,primary_test_eligibility
0,DIV2K,yes,no,yes,no,exclude
1,Flickr2K,yes,no,yes,no,exclude
2,BSD400 / BSD family,no,BSDS500 colour and gray,yes,BSD68 / CBSD68,exclude
3,Waterloo Exploration Database,no,no,yes,no,exclude
4,LIVE1,no,yes,no,yes,exclude
5,Classic5,no,gray benchmark,no,JPEG deblocking benchmark,exclude
6,ICB,no,yes,no,no,exclude
7,Kodak24,no,no,no,yes,exclude
8,McMaster18,no,no,no,yes,exclude
9,Set12,no,no,no,yes,exclude


PASS — exactly one strict primary cohort is eligible, with the stated caveat.


## 3. Exact sealed cohort

All 40 images from the named 8-bit RGB 2400 × 2400 archive are selected by a content-independent decode rule. The archive and member hashes are intentionally deferred to stage 05C because the bytes have not yet been downloaded. The independent run remains locked until that receipt and duplicate audit exist.

In [4]:
dataset = protocol['sealed_primary_dataset']
dataset_view = pd.DataFrame([{
    'dataset': dataset['name'],
    'version': dataset['version'],
    'archive': dataset['archive_name'],
    'expected_sources': dataset['expected_sources'],
    'format': dataset['expected_format'],
    'mode': dataset['expected_mode'],
    'bits/channel': dataset['expected_bit_depth_per_channel'],
    'dimensions': f"{dataset['expected_width']}x{dataset['expected_height']}",
    'archive_sha256_now': dataset['archive_sha256'],
}])
display(dataset_view)
assert dataset['expected_sources'] == 40
assert dataset['archive_name'] == 'SAMPLING_8BIT_RGB_2400x2400.tar.bz2'
assert dataset['archive_sha256'] is None
print('PASS — dataset identity and selection rule are frozen; byte receipt is still pending.')

,dataset,version,archive,expected_sources,format,mode,bits/channel,dimensions,archive_sha256_now
0,TESTIMAGES/SAMPLING,4.0.000,SAMPLING_8BIT_RGB_2400x2400.tar.bz2,40,PNG,RGB,8,2400x2400,None


PASS — dataset identity and selection rule are frozen; byte receipt is still pending.


## 4. Frozen acquisition ladder and reconstruction methods

In [5]:
chains = pd.DataFrame(protocol['simulation']['acquisition_chains'])
display(chains)
display(pd.DataFrame({'reconstruction_method': protocol['reconstruction_methods']}))

assert len(chains) == 7
assert chains.id.is_unique
assert chains.loc[chains.role.eq('primary_anchor'), 'id'].tolist() == ['j75_b16_n2']
assert set(chains.jpeg_quality.dropna().astype(int)) >= {50, 75, 90}
assert len(protocol['reconstruction_methods']) == 6
print('PASS — seven unique acquisition chains and six fixed reconstruction outputs.')

,id,true_blur_sigma,noise_std,codec,jpeg_quality,role
0,q8_b16_n2,1.6,0.007843,quantized_8bit,NaN,uncompressed_negative_control
1,j90_b16_n2,1.6,0.007843,jpeg,90.0,secondary_codec_strength
2,j75_b16_n2,1.6,0.007843,jpeg,75.0,primary_anchor
3,j50_b16_n2,1.6,0.007843,jpeg,50.0,secondary_codec_strength
4,j75_b12_n2,1.2,0.007843,jpeg,75.0,secondary_blur_strength
5,j75_b20_n2,2.0,0.007843,jpeg,75.0,secondary_blur_strength
6,j75_b16_n5,1.6,0.019608,jpeg,75.0,secondary_noise_strength


,reconstruction_method
0,observed
1,gradient_nominal
2,dpir_nominal
3,fbcnn
4,fbcnn_gradient_nominal
5,fbcnn_dpir_nominal


PASS — seven unique acquisition chains and six fixed reconstruction outputs.


## 5. Primary hypotheses and inference

The source image is the replication unit. Patch rows remain nested measurements. The two primary tests share a Holm correction and require both statistical evidence and a 5% practical improvement.

In [6]:
hypotheses = pd.DataFrame(protocol['primary_hypotheses'])
display(hypotheses[['id', 'condition', 'endpoint', 'contrast', 'direction', 'practical_gate']])
stats = protocol['statistical_plan']
display(pd.DataFrame([
    {'item': 'independent unit', 'value': stats['independent_unit']},
    {'item': 'patches independent?', 'value': stats['patches_as_independent_replicates']},
    {'item': 'bootstrap', 'value': stats['bootstrap']},
    {'item': 'randomization test', 'value': stats['randomization_test']},
    {'item': 'multiplicity', 'value': stats['multiplicity']},
]))
assert hypotheses.id.tolist() == ['H1_reconstruction', 'H2_selection']
assert stats['patches_as_independent_replicates'] is False
assert 'Holm' in stats['multiplicity']
print('PASS — two source-level primary gates are fixed.')

,id,condition,endpoint,contrast,direction,practical_gate
0,H1_reconstruction,j75_b16_n2,mean detail MSE over the 512x512 evaluation re...,fbcnn_dpir_nominal minus dpir_nominal,negative is favorable,At least 5% relative reduction versus dpir_nom...
1,H2_selection,j75_b16_n2,retained-patch detail MSE,operator_spread_detail risk minus image_transf...,negative is favorable,At least 5% relative risk reduction versus ima...


,item,value
0,independent unit,source image
1,patches independent?,False
2,bootstrap,"10,000 source-level paired percentile resample..."
3,randomization test,"100,000 source-level paired sign flips using s..."
4,multiplicity,Holm correction across H1 and H2 only


PASS — two source-level primary gates are fixed.


## 6. Stronger trained image-only comparator and calibration separation

PatchErrorNet sees only the observed and reconstructed image context. Training, early stopping and score calibration use disjoint source-ID blocks; the sealed cohort participates in none of them.

In [7]:
partitions = protocol['development_partitions']
comparator = protocol['trained_image_only_comparator']
display(pd.DataFrame([
    {'partition': 'fit', 'sources': partitions['trained_comparator_fit_sources'], 'role': 'learn weights'},
    {'partition': 'early stop', 'sources': partitions['trained_comparator_early_stop_sources'], 'role': 'choose epoch'},
    {'partition': 'calibration', 'sources': partitions['score_calibration_sources'], 'role': 'fit isotonic mappings'},
    {'partition': 'engineering canary', 'sources': ', '.join(partitions['engineering_canary_sources']), 'role': 'engineering only'},
    {'partition': 'independent test', 'sources': 'eligible TESTIMAGES/SAMPLING sources', 'role': 'sealed evaluation'},
]))
display(pd.DataFrame([
    {'property': 'backbone', 'value': comparator['backbone']},
    {'property': 'ensemble members', 'value': comparator['ensemble_members']},
    {'property': 'information', 'value': comparator['information']},
    {'property': 'target', 'value': comparator['target']},
    {'property': 'early stopping', 'value': comparator['early_stopping']},
]))

assert comparator['ensemble_members'] == 5
assert 'no operator variants' in comparator['information']
assert partitions['trained_comparator_fit_sources'] == '0805-0856 inclusive (52 sources)'
assert partitions['score_calibration_sources'] == '0869-0900 inclusive (32 sources)'
print('PASS — training, early stopping, calibration and independent evaluation are source-separated.')

,partition,sources,role
0,fit,0805-0856 inclusive (52 sources),learn weights
1,early stop,0857-0868 inclusive (12 sources),choose epoch
2,calibration,0869-0900 inclusive (32 sources),fit isotonic mappings
3,engineering canary,"0805, 0806",engineering only
4,independent test,eligible TESTIMAGES/SAMPLING sources,sealed evaluation


,property,value
0,backbone,ResNet-18 from scratch with a 6-channel first ...
1,ensemble members,5
2,information,64x64 context from the supplied observation an...
3,target,Whether the centre 16x16 patch has detail RMSE...
4,early stopping,Patience 5 on source-macro Brier score in sour...


PASS — training, early stopping, calibration and independent evaluation are source-separated.


## 7. Execution barriers

In [8]:
barriers = pd.DataFrame(protocol['execution_barriers'])
display(barriers)
requirements = pd.DataFrame({
    'required_before_independent_inference': protocol['readiness_requirements_before_independent_inference']
})
display(requirements)

assert protocol['test_results_inspected'] is False
assert protocol['independent_test_run_authorized'] is False
assert protocol['current_status']['independent_test_run'] is False
print('PASS — independent inference and performance inspection remain blocked.')

,stage,allowed,forbidden
0,05A-05B,"Protocol freeze, documented exposure audit, st...",Download or inspect independent reconstruction...
1,05C,Acquire and hash data; run byte/decode/duplica...,Run inference on TESTIMAGES or change scientif...
2,05D,Locked full run after every readiness assertio...,"Interim test peeking, method tuning, threshold..."
3,05E-05F,"One-time unsealing, calibration application, f...","Changing hypotheses, comparators, exclusions o..."


,required_before_independent_inference
0,Archive SHA-256 and byte count receipt exists.
1,Exactly 40 eligible source records exist befor...
2,All decoded sources are RGB 2400x2400 8-bit PNGs.
3,Exact and near-duplicate audit against all dev...
4,At least 36 primary sources remain eligible.
5,Pinned DPIR and FBCNN source and checkpoint di...
6,PatchErrorNet training and early stopping are ...
7,All isotonic mappings are fitted on DIV2K 0869...
8,The DIV2K 0805-0806 engineering canary passes ...
9,The run code verifies this protocol's SHA-256 ...


PASS — independent inference and performance inspection remain blocked.


## 8. Independent machine validation and freeze hashes

The repository validator repeats the structural assertions and checks that no independent performance artifacts exist. The recorded receipt binds the protocol, audit, explanatory document and validator by SHA-256.

In [9]:
spec = importlib.util.spec_from_file_location('validate_independent_05_protocol', VALIDATOR_PATH)
validator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validator)
validation = validator.validate()

for key in ('protocol_sha256', 'audit_sha256', 'document_sha256', 'validator_sha256'):
    assert validation[key] == receipt[key], f'Receipt mismatch for {key}'

display(pd.DataFrame([validation]).T.rename(columns={0: 'value'}))
print('PASS — validator passed, hashes match the freeze receipt, and no independent outcomes were found.')

,value
status,pass
experiment_id,independent_05
validated_stage,05A_protocol_freeze_and_05B_documented_overlap...
protocol_sha256,b92c6cf73e05f60dc1edb3a31d88d91623e9ae0cf63d62...
audit_sha256,47b4bb50257e5e7e1874231bcb359dfa44d3a117c4a495...
document_sha256,08b09811ed4134ec30a189f9789f193e4dc39f89c089ef...
validator_sha256,238e935750ac5d56d61ef2811add59e38abec7c06666b4...
audit_rows,11
excluded_audit_rows,10
eligible_primary_cohort,TESTIMAGES/SAMPLING


PASS — validator passed, hashes match the freeze receipt, and no independent outcomes were found.


## Audit conclusion

Stages 05A and 05B pass. The protocol and documented exposure audit are frozen, and no independent performance artifact exists. The next permitted work is stage 05C: acquire and hash the named archive, complete byte/decode/duplicate checks, implement the locked runner and trained comparator, fit calibration mappings on their fixed development split, and run an engineering canary on DIV2K 0805–0806 only.

**Do not run TESTIMAGES inference until every readiness assertion is true and the protocol's authorization state is advanced in a versioned, outcome-blind stage transition.**